In [5]:
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import tensorflow as tf
import numpy as np
import cv2
import json
import io
from PIL import Image
import uvicorn

In [11]:
def preprocess_image(image_bytes, target_size=(224, 224)):
    """
    Preprocess an image from bytes for model prediction.
    """
    image = Image.open(io.BytesIO(image_bytes))

    if image.mode != 'RGB':
        image = image.convert('RGB')  # This line must be indented

    image = np.array(image)
    resized_image = cv2.resize(image, target_size)
    rescaled_image = resized_image.astype('float32') / 255.0
    rescaled_image = np.expand_dims(rescaled_image, axis=0)

    return rescaled_image

In [12]:
app = FastAPI()

model = tf.keras.models.load_model("Image_Detection_Model.h5")


CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

@app.post("/predict_tumor")
async def predict(file: UploadFile = File(...)):
    """
    Make a prediction for the uploaded MRI image.
    """
    try:
       
        contents = await file.read()

        
        image = preprocess_image(contents)

        
        prediction = model.predict(image)
        predicted_class_index = np.argmax(prediction)  
        
        
        prediction_class = CLASS_NAMES[predicted_class_index]

        return {"prediction": prediction_class}
        
    except Exception as e:
        return JSONResponse({
            "success": False,
            "error": str(e)
        }, status_code=500)

In [ ]:
#http://localhost:8000/docs
import nest_asyncio

nest_asyncio.apply()

if __name__ == "__main__":

    uvicorn.run(app, host="0.0.0.0", port=8000)

INFO:     Started server process [2336]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:50424 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:50424 - "GET /openapi.json HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
INFO:     127.0.0.1:50425 - "POST /predict_tumor HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step
INFO:     127.0.0.1:50486 - "POST /predict_tumor HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
INFO:     127.0.0.1:50525 - "POST /predict_tumor HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
INFO:     127.0.0.1:50526 - "POST /predict_tumor HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step
INFO:     127.0.0.1:50536 - "POST /predict_tumor HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
INFO:     127.0.0.1:50537 - "POST /predict_tumor HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
INFO:     127.0.0.1:50556 - "POST /predict_tumor HTTP/1.1" 200 OK
